In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

# Initialize Faker
fake = Faker()

def generate_realistic_sales(num_records=2000):
    data = []
    
    # --- Configuration: Real-world Pricing & Margins ---
    categories = {
        'Electronics': {'price': (15000, 150000), 'margin': 0.15},
        'Apparel':     {'price': (1000, 8000),     'margin': 0.40},
        'Grocery':     {'price': (50, 1500),       'margin': 0.10},
        'Dairy':       {'price': (80, 500),        'margin': 0.08},
        'Home Decor':  {'price': (2000, 25000),    'margin': 0.35}
    }
    
    payment_methods = ['Cash', 'Credit Card', 'Debit Card', 'JazzCash', 'EasyPaisa']
    
    # Start generating from 30 days ago till today
    start_date = datetime.now() - timedelta(days=30)

    for i in range(num_records):
        # 1. Realistic Timestamp (Peak Hours Logic)
        # Retail mein peak 6 PM - 9 PM hota hai
        current_date = start_date + timedelta(days=random.randint(0, 30))
        hour_weights = [1,1,1,1,1,2,5,10,15,12,10,15,20,30,40,35,25,15,10,5,2,1,1,1] # 24 hours weights
        hour = random.choices(range(24), weights=hour_weights)[0]
        timestamp = current_date.replace(hour=hour, minute=random.randint(0, 59), second=random.randint(0, 59))
        
        # 2. Pick Category and Product
        cat_name = random.choice(list(categories.keys()))
        min_p, max_p = categories[cat_name]['price']
        
        # 3. Pricing Logic
        unit_price = round(random.uniform(min_p, max_p), 2)
        qty = random.choices([1, 2, 3, 4, 5], weights=[70, 15, 8, 5, 2])[0]
        
        # 4. Discount Logic (15% transactions have discounts)
        has_discount = random.random() < 0.15
        discount_amt = round(unit_price * random.uniform(0.05, 0.20), 2) if has_discount else 0
        
        # 5. COGS calculation (Unit Price - Margin)
        margin_pct = categories[cat_name]['margin']
        cogs_per_unit = round(unit_price * (1 - margin_pct), 2)
        
        # 6. Transaction Details
        txn_id = f"TXN-{2024}{i+1000:05d}"
        sku_id = f"SKU-{cat_name[:3].upper()}-{random.randint(100, 999)}"
        
        data.append([
            txn_id, 
            timestamp, 
            sku_id, 
            cat_name, 
            qty, 
            unit_price, 
            discount_amt, 
            cogs_per_unit, 
            random.choice(payment_methods)
        ])

    # Create DataFrame
    df = pd.DataFrame(data, columns=[
        'Transaction_ID', 'Timestamp', 'SKU_ID', 'Category_ID', 
        'Quantity_Sold', 'Unit_Price', 'Discount_Applied', 
        'COGS_Per_Unit', 'Payment_Method'
    ])
    
    # Sort by time to make it look like a real ledger
    df = df.sort_values(by='Timestamp').reset_index(drop=True)
    return df

# Generate and Save
sales_data = generate_realistic_sales(1000) # Generating 5000 rows
sales_data.to_csv('../Data/Sales_Data.csv', index=False)




In [3]:
sales_data.head()

,Transaction_ID,Timestamp,SKU_ID,Category_ID,Quantity_Sold,Unit_Price,Discount_Applied,COGS_Per_Unit,Payment_Method
0,TXN-202402250,2026-01-27 01:21:39.711408,SKU-HOM-837,Home Decor,1,5813.45,0.00,3778.74,Cash
1,TXN-202401438,2026-01-27 01:43:24.711408,SKU-GRO-369,Grocery,2,1279.35,0.00,1151.41,EasyPaisa
2,TXN-202403238,2026-01-27 03:15:45.711408,SKU-DAI-584,Dairy,1,471.29,33.49,433.59,Credit Card
3,TXN-202401132,2026-01-27 05:22:46.711408,SKU-GRO-578,Grocery,1,1433.94,95.46,1290.55,Debit Card
4,TXN-202401235,2026-01-27 06:17:32.711408,SKU-APP-945,Apparel,2,2447.84,0.00,1468.70,JazzCash
